In [1]:
import os
import re
import string
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from underthesea import word_tokenize
from gensim.models import Word2Vec
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

In [2]:
DATA_FILE = "../output/news_labeled_clean.csv"
df = pd.read_csv(DATA_FILE)
df = df.sample(n=50_000, random_state=42).reset_index(drop=True)
print(df.shape)
print(df.columns.tolist())
df.head(3)

(50000, 10)
['id', 'url', 'category', 'label', 'title', 'desc', 'text', 'source', 'author', 'crawled_at']


,id,url,category,label,title,desc,text,source,author,crawled_at
0,103003,https://tienphong.vn/bi-thu-t-u-doan-tang-qua-doi-hinh-tiep-suc-mua-thi-vinh-phuc-ninh-binh-post1451781.tpo,giới trẻ,0,"Bí thư T.Ư Đoàn tặng quà đội hình tiếp sức mùa thi Vĩnh Phúc, Ninh Bình","Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hình thanh niên tình nguyện Tiếp sức mùa thi năm 2022 tại điểm thi Trường T...","Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hình thanh niên tình nguyện Tiếp sức mùa thi năm 2022 tại điểm thi Trường T...",tienphong,Xuân Tùng,2022-07-07 14:52:37.891739
1,13977,https://www.24h.com.vn/ban-tre-cuoc-song/do-khoc-do-cuoi-voi-phan-ung-cua-be-gai-khi-bai-kiem-tra-bi-cho-cung-can-nat-c64a1369635.html,bạn trẻ - cuộc sống,0,Dở khóc dở cười với phản ứng của bé gái khi bài kiểm tra bị chó cưng cắn nát,"Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bên đống hỗn độn mà chó cưng đã tạo ra khi hai mẹ con vắng nhà. Được biết, bé g...","Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bên đống hỗn độn mà chó cưng đã tạo ra khi hai mẹ con vắng nhà. Được biết, bé g...",24h.com.vn,Theo Đinh Kim (T/h) (Đời sống & Pháp luật),2022-06-17 17:24:44.587671
2,62431,https://vov.vn/xa-hoi/tin-24h/hai-hoc-sinh-duoi-nuoc-khi-tam-bien-o-quang-tri-post953629.vov,xã hội,0,Hai học sinh đuối nước khi tắm biển ở Quảng Trị,"Trước đó, cuối buổi chiều 29/6, nhóm 3 học sinh ở xã Phong Bình, huyện Gio Linh rủ nhau tắm biển ở khu vực thôn 5, xã Gio Hải, huyện Gio Linh thì bị đuối nước. Một học sinh bơi được vào bờ và tri ...","Trước đó, cuối buổi chiều 29/6, nhóm 3 học sinh ở xã Phong Bình, huyện Gio Linh rủ nhau tắm biển ở khu vực thôn 5, xã Gio Hải, huyện Gio Linh thì bị đuối nước. Một học sinh bơi được vào bờ và tri ...",vov.vn,Đình Thiệu/VOV-Miền Trung,2022-06-30 13:04:53.246926


In [3]:
df["title_raw"] = df["title"].fillna("").astype(str)
df["desc_raw"] = df["desc"].fillna("").astype(str)
df["text_raw"] = df["text"].fillna("").astype(str)

df[["title_raw", "desc_raw", "text_raw"]].head(2)

,title_raw,desc_raw,text_raw
0,"Bí thư T.Ư Đoàn tặng quà đội hình tiếp sức mùa thi Vĩnh Phúc, Ninh Bình","Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hình thanh niên tình nguyện Tiếp sức mùa thi năm 2022 tại điểm thi Trường T...","Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hình thanh niên tình nguyện Tiếp sức mùa thi năm 2022 tại điểm thi Trường T..."
1,Dở khóc dở cười với phản ứng của bé gái khi bài kiểm tra bị chó cưng cắn nát,"Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bên đống hỗn độn mà chó cưng đã tạo ra khi hai mẹ con vắng nhà. Được biết, bé g...","Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bên đống hỗn độn mà chó cưng đã tạo ra khi hai mẹ con vắng nhà. Được biết, bé g..."


In [4]:
df["model_input_raw"] = (
    df["title_raw"].str.strip() + " " + df["text_raw"].str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

df[["title_raw", "text_raw", "model_input_raw"]].head(2)

,title_raw,text_raw,model_input_raw
0,"Bí thư T.Ư Đoàn tặng quà đội hình tiếp sức mùa thi Vĩnh Phúc, Ninh Bình","Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hình thanh niên tình nguyện Tiếp sức mùa thi năm 2022 tại điểm thi Trường T...","Bí thư T.Ư Đoàn tặng quà đội hình tiếp sức mùa thi Vĩnh Phúc, Ninh Bình Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hìn..."
1,Dở khóc dở cười với phản ứng của bé gái khi bài kiểm tra bị chó cưng cắn nát,"Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bên đống hỗn độn mà chó cưng đã tạo ra khi hai mẹ con vắng nhà. Được biết, bé g...","Dở khóc dở cười với phản ứng của bé gái khi bài kiểm tra bị chó cưng cắn nát Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bê..."


In [5]:
vi_stopwords = {
    "và", "là", "của", "có", "được", "trong", "một", "những", "các", "cho", "với",
    "khi", "đã", "đang", "theo", "về", "ở", "ra", "này", "đó", "thì", "lại", "từ",
    "đến", "sau", "trên", "dưới", "tại", "bị", "do", "nên", "vẫn", "rằng", "như",
    "để", "hay", "vào", "hơn", "ít", "nhiều", "rất", "cũng", "mới"
}

In [6]:
def preprocess_vietnamese_text(text):
    text = str(text).lower().strip()
    
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(rf"[{re.escape(string.punctuation)}]", " ", text)
    text = re.sub(r"[“”‘’…–—]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    
    tokens = text.split()
    tokens = [tok for tok in tokens if tok not in vi_stopwords]
    text = " ".join(tokens)
    
    return text

df["model_input_clean"] = df["model_input_raw"].apply(
    lambda x: preprocess_vietnamese_text(x)
)

df[["model_input_raw", "model_input_clean"]].head(3)

,model_input_raw,model_input_clean
0,"Bí thư T.Ư Đoàn tặng quà đội hình tiếp sức mùa thi Vĩnh Phúc, Ninh Bình Tại Vĩnh Phúc, đoàn công tác do anh Nguyễn Tường Lâm, Bí thư T.Ư Đoàn làm trưởng đoàn đã thăm, tặng quà và động viên đội hìn...",bí thư t ư đoàn tặng quà đội hình tiếp sức mùa thi vĩnh phúc ninh bình vĩnh phúc đoàn công tác anh nguyễn tường lâm bí thư t ư đoàn làm trưởng đoàn thăm tặng quà động viên đội hình thanh niên tình...
1,"Dở khóc dở cười với phản ứng của bé gái khi bài kiểm tra bị chó cưng cắn nát Một bà mẹ ở Yên Đài, tỉnh Sơn Đông (Trung Quốc) mới đây đã đăng tải đoạn video ghi lại cảnh con gái ngồi khóc nức nở bê...",dở khóc dở cười phản ứng bé gái bài kiểm tra chó cưng cắn nát bà mẹ yên đài tỉnh sơn đông trung quốc đây đăng tải đoạn video ghi cảnh con gái ngồi khóc nức nở bên đống hỗn độn mà chó cưng tạo hai ...
2,"Hai học sinh đuối nước khi tắm biển ở Quảng Trị Trước đó, cuối buổi chiều 29/6, nhóm 3 học sinh ở xã Phong Bình, huyện Gio Linh rủ nhau tắm biển ở khu vực thôn 5, xã Gio Hải, huyện Gio Linh thì bị...",hai học sinh đuối nước tắm biển quảng trị trước cuối buổi chiều nhóm học sinh xã phong bình huyện gio linh rủ nhau tắm biển khu vực thôn xã gio hải huyện gio linh đuối nước học sinh bơi bờ tri hô ...


In [7]:
X = df["model_input_clean"]
y = df["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print("Train size:", len(X_train))
print("Test size :", len(X_test))
print("Train label distribution:")
print(y_train.value_counts(normalize=True).sort_index())
print("Test label distribution:")
print(y_test.value_counts(normalize=True).sort_index())

Train size: 40000
Test size : 10000
Train label distribution:
label
0    0.910175
1    0.089825
Name: proportion, dtype: float64
Test label distribution:
label
0    0.9102
1    0.0898
Name: proportion, dtype: float64


In [8]:
CACHE_PATH = "segmented_cache.csv"

def segment_series(text_series: pd.Series) -> pd.Series:
    """Áp dụng word_tokenize cho từng dòng, trả về Series đã tách từ."""
    tqdm.pandas(desc="Word segmentation")
    return text_series.progress_apply(
        lambda text: word_tokenize(str(text), format="text")
    )

if os.path.exists(CACHE_PATH):
    print(f"Cache tồn tại → load từ {CACHE_PATH}")
    cache_df    = pd.read_csv(CACHE_PATH)
    X_train_seg = pd.Series(cache_df[cache_df["split"] == "train"]["text_seg"].values)
    X_test_seg  = pd.Series(cache_df[cache_df["split"] == "test" ]["text_seg"].values)
    print(f"Load xong: train={len(X_train_seg)}, test={len(X_test_seg)}")
else:
    print("Chưa có cache → bắt đầu tách từ...")
    X_train_seg = segment_series(X_train.reset_index(drop=True))
    X_test_seg  = segment_series(X_test.reset_index(drop=True))

    cache_df = pd.DataFrame({
        "split":    ["train"] * len(X_train_seg) + ["test"] * len(X_test_seg),
        "text_seg": X_train_seg.tolist() + X_test_seg.tolist(),
    })
    cache_df.to_csv(CACHE_PATH, index=False)
    print(f"Đã lưu cache → {CACHE_PATH}")

# Demo kiểm tra nhanh
sample = "Chàng trai 9X Quảng Trị khởi nghiệp từ nấm sò"
print(f"\nDemo: '{sample}'")
print(f"→ '{word_tokenize(sample, format='text')}'")

Cache tồn tại → load từ segmented_cache.csv
Load xong: train=40000, test=10000

Demo: 'Chàng trai 9X Quảng Trị khởi nghiệp từ nấm sò'
→ 'Chàng trai 9X Quảng_Trị khởi_nghiệp từ nấm sò'


## Tokenize: string → list of tokens

In [9]:
def to_token_list(text):
    return str(text).split()

train_tokens = X_train_seg.apply(to_token_list).tolist()
test_tokens  = X_test_seg.apply(to_token_list).tolist()

print(f"Train: {len(train_tokens)} docs")
print(f"Sample doc (10 tokens): {train_tokens[0][:10]}")

Train: 40000 docs
Sample doc (10 tokens): ['giúp', 'hai', 'mẹ_con', 'quên', 'giấy_tờ', 'sân_bay', 'tài_xế', 'buồn_lòng', 'vì', 'ngờ_vực']


## Train Word2Vec trên toàn corpus

In [10]:
all_tokens = train_tokens + test_tokens  # unsupervised → dùng cả train+test

w2v_model = Word2Vec(
    sentences=all_tokens,
    vector_size=100,   # số chiều vector
    window=5,          # context window
    min_count=3,       # bỏ từ xuất hiện < 3 lần
    workers=4,
    epochs=10,
    seed=42
)

print(f"Vocab size : {len(w2v_model.wv)}")
print(f"Vector size: {w2v_model.vector_size}")

Vocab size : 98472
Vector size: 100


## Hàm trung bình vector (document embedding)

In [11]:
def doc_to_vec(tokens, model, vector_size):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        return np.zeros(vector_size)
    return np.mean(vecs, axis=0)

## Tạo ma trận feature

In [12]:
vsz = w2v_model.vector_size

X_train_w2v = np.array([doc_to_vec(t, w2v_model, vsz) for t in train_tokens])
X_test_w2v  = np.array([doc_to_vec(t, w2v_model, vsz) for t in test_tokens])

print(f"X_train_w2v: {X_train_w2v.shape}")
print(f"X_test_w2v : {X_test_w2v.shape}")

X_train_w2v: (40000, 100)
X_test_w2v : (10000, 100)


## Train LR + SVM với Word2Vec vectors

In [13]:
models_w2v = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM':                LinearSVC(random_state=42),
}

results_w2v = []
for model_name, model in models_w2v.items():
    model.fit(X_train_w2v, y_train)
    y_pred = model.predict(X_test_w2v)

    acc = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average='binary', pos_label=1, zero_division=0
    )
    results_w2v.append({
        'vectorizer':       'Word2Vec (avg)',
        'model':            model_name,
        'accuracy':         round(acc, 4),
        'precision_class1': round(precision, 4),
        'recall_class1':    round(recall, 4),
        'f1_class1':        round(f1, 4),
    })

results_w2v_df = (pd.DataFrame(results_w2v)
                    .sort_values(by='f1_class1', ascending=False)
                    .reset_index(drop=True))
print("Kết quả Word2Vec + Classifiers:")
results_w2v_df

Kết quả Word2Vec + Classifiers:


,vectorizer,model,accuracy,precision_class1,recall_class1,f1_class1
0,Word2Vec (avg),LogisticRegression,0.9415,0.7078,0.5935,0.6457
1,Word2Vec (avg),SVM,0.9421,0.7295,0.5646,0.6365


## So sánh với TF-IDF

In [14]:
tfidf_best = pd.DataFrame([
    {'vectorizer': 'TF-IDF (1,2)', 'model': 'SVM',
     'accuracy': 0.9478, 'precision_class1': 0.7467, 'recall_class1': 0.6336, 'f1_class1': 0.6855},
    {'vectorizer': 'TF-IDF (1,2)', 'model': 'LogisticRegression',
     'accuracy': 0.9478, 'precision_class1': 0.7733, 'recall_class1': 0.5924, 'f1_class1': 0.6709},
    {'vectorizer': 'TF-IDF (seg)', 'model': 'SVM',
     'accuracy': 0.9461, 'precision_class1': 0.7429, 'recall_class1': 0.6114, 'f1_class1': 0.6707},
])

compare_df = (pd.concat([tfidf_best, results_w2v_df], ignore_index=True)
                .sort_values(by='f1_class1', ascending=False)
                .reset_index(drop=True))

print("So sánh TF-IDF vs Word2Vec:")
compare_df

So sánh TF-IDF vs Word2Vec:


,vectorizer,model,accuracy,precision_class1,recall_class1,f1_class1
0,"TF-IDF (1,2)",SVM,0.9478,0.7467,0.6336,0.6855
1,"TF-IDF (1,2)",LogisticRegression,0.9478,0.7733,0.5924,0.6709
2,TF-IDF (seg),SVM,0.9461,0.7429,0.6114,0.6707
3,Word2Vec (avg),LogisticRegression,0.9415,0.7078,0.5935,0.6457
4,Word2Vec (avg),SVM,0.9421,0.7295,0.5646,0.6365


## Tuning tham số Word2Vec

In [15]:
configs = [
    {'vector_size': 100, 'window':  5, 'min_count': 3, 'epochs': 10},  # baseline
    {'vector_size': 200, 'window':  5, 'min_count': 3, 'epochs': 10},  # dim lớn hơn
    {'vector_size': 100, 'window': 10, 'min_count': 3, 'epochs': 10},  # window rộng hơn
    {'vector_size': 100, 'window':  5, 'min_count': 1, 'epochs': 10},  # giữ từ hiếm
    {'vector_size': 100, 'window':  5, 'min_count': 3, 'epochs': 20},  # train lâu hơn
    {'vector_size': 200, 'window': 10, 'min_count': 3, 'epochs': 20},  # kết hợp tốt nhất
]

tuning_results = []
for cfg in configs:
    w2v = Word2Vec(
        sentences=all_tokens,
        vector_size=cfg['vector_size'],
        window=cfg['window'],
        min_count=cfg['min_count'],
        workers=4,
        epochs=cfg['epochs'],
        seed=42
    )
    Xtr = np.array([doc_to_vec(t, w2v, w2v.vector_size) for t in train_tokens])
    Xte = np.array([doc_to_vec(t, w2v, w2v.vector_size) for t in test_tokens])

    for mname, clf in [('LR',  LogisticRegression(max_iter=1000, random_state=42)),
                       ('SVM', LinearSVC(random_state=42))]:
        clf.fit(Xtr, y_train)
        yp = clf.predict(Xte)
        acc = accuracy_score(y_test, yp)
        _, _, f1, _ = precision_recall_fscore_support(
            y_test, yp, average='binary', pos_label=1, zero_division=0
        )
        tuning_results.append({**cfg, 'model': mname,
                                'accuracy': round(acc, 4),
                                'f1_class1': round(f1, 4)})

tuning_df = (pd.DataFrame(tuning_results)
               .sort_values(by='f1_class1', ascending=False)
               .reset_index(drop=True))
print("Kết quả tuning Word2Vec:")
tuning_df

Kết quả tuning Word2Vec:


,vector_size,window,min_count,epochs,model,accuracy,f1_class1
0,200,10,3,20,SVM,0.9439,0.6556
1,200,10,3,20,LR,0.9424,0.6547
2,100,10,3,10,LR,0.9420,0.6498
3,100,10,3,10,SVM,0.9433,0.6467
4,200,5,3,10,LR,0.9414,0.6466
5,200,5,3,10,SVM,0.9422,0.6436
6,100,5,3,10,LR,0.9413,0.6432
7,100,5,3,10,SVM,0.9425,0.6417
8,100,5,1,10,LR,0.9409,0.6407
9,100,5,1,10,SVM,0.9420,0.6402
